In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

In [3]:
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print(f'Using device: MPS (Apple Silicon GPU)')
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'Using device: CUDA (NVIDIA GPU)')
else:
    device = torch.device('cpu')
    print(f'Using device: CPU')

print(f'PyTorch version: {torch.__version__}')

Using device: CPU
PyTorch version: 2.11.0+cpu


In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d shayanfazeli/heartbeat

In [ ]:
!unzip heartbeat.zip

In [ ]:
df_train = pd.read_csv('mitbih_train.csv', header=None)
print(f"Train data loaded: {df_train.shape}")

df_test = pd.read_csv('mitbih_test.csv', header=None)
print(f"Test data loaded: {df_test.shape}")

In [ ]:
x_train = df_train.iloc[:, :-1].values
y_train = df_train.iloc[:, -1].values.astype(int)

x_test = df_test.iloc[:, :-1].values
y_test = df_test.iloc[:, -1].values.astype(int)

In [ ]:
!pip install wfdb

In [ ]:
import wfdb
import numpy as np
from scipy.signal import resample
from collections import Counter

AAMI_MAP = {
    'N': 0, '.': 0, 'n': 0,
    'A': 1, 'a': 1, 'S': 1, 'J': 1, 'e': 1, 'j': 1,
    'V': 2, 'E': 2,
    'F': 3,
    'Q': 4, '?': 4, '/': 4, 'f': 4, 'U': 4
}

WINDOW = 187
HALF = WINDOW // 2

def extract_segments(signal, r_peaks, annotations, target_fs, source_fs):
  if source_fs != target_fs:
    factor = target_fs / source_fs
    signal = resample(signal, int(len(signal) * factor))
    r_peaks = (r_peaks * factor).astype(int)

  segments, labels = [], []

  for peak, symbol in zip(r_peaks, annotations):
    if symbol not in AAMI_MAP:
      continue

    start = peak - HALF
    end = start + WINDOW

    if start < 0 or end > len(signal):
      continue

    window = signal[start:end].astype(np.float32)

    w_min, w_max = window.min(), window.max()
    if w_max - w_min < 1e-6:
      continue

    window = (window - w_min) / (w_max - w_min)

    segments.append(window)
    labels.append(AAMI_MAP[symbol])

  return np.array(segments, dtype=np.float32), np.array(labels, dtype=np.int64)


In [ ]:
def load_mitbih_sva(data_path="mitdbsvdb"):
    """
    MIT-BIH Supraventricular Arrhythmia Database.
    78 half-hour recordings, rich in SVT/PAC (class 1).
    Download: https://physionet.org/content/svdb/1.0.0/

    Args:
        data_path : Local folder where the database was downloaded,
                    OR pass 'mitdbsvdb' to stream directly from PhysioNet
                    (requires internet in Colab)
    """
    # Record IDs for this database
    record_ids = [
        '800', '801', '802', '803', '804', '805', '806', '807',
        '808', '809', '810', '811', '812', '820', '821', '822',
        '823', '824', '825', '826', '827', '828', '829', '840',
        '841', '842', '843', '844', '845', '846', '847', '848',
        '849', '850', '851', '852', '853', '854', '855', '856',
        '857', '858', '859', '860', '861', '862', '863', '864',
        '865', '866', '867', '868', '869', '870', '871', '872',
        '873', '874', '875', '876', '877', '878', '879', '880',
        '881', '882', '883', '884', '885', '886', '887', '888',
        '889', '890', '891', '892'
    ]
    return _load_physionet_db(record_ids, db_name="svdb", source_fs=128)


def load_incart(data_path="incartdb"):
    """
    St. Petersburg INCART 12-lead Arrhythmia Database.
    75 annotated Holter recordings, 30 min each, 257 Hz.
    Uses Lead II (channel index 1) to match MIT-BIH.
    Download: https://physionet.org/content/incartdb/1.0.0/

    Args:
        data_path : Local folder or 'incartdb/1.0.0' to stream from PhysioNet
    """
    record_ids = [f"I{str(i).zfill(2)}" for i in range(1, 76)]  # I01–I75
    # PhysioNet new URL structure requires versioned path
    return _load_physionet_db(record_ids, db_name="incartdb/1.0.0",
                               source_fs=257, lead_index=1)


def load_estdb(data_path="estdb"):
    """
    European ST-T Database.
    90 annotated 2-hour recordings, 250 Hz.
    Good morphological variation from European patient cohort.
    Download: https://physionet.org/content/edb/1.0.0/

    Args:
        data_path : Local folder or 'edb/1.0.0' to stream from PhysioNet
    """
    record_ids = [
        'e0103', 'e0104', 'e0105', 'e0106', 'e0107', 'e0108',
        'e0110', 'e0111', 'e0112', 'e0113', 'e0114', 'e0115',
        'e0116', 'e0118', 'e0119', 'e0121', 'e0122', 'e0123',
        'e0124', 'e0125', 'e0126', 'e0127', 'e0129', 'e0133',
        'e0136', 'e0139', 'e0147', 'e0148', 'e0151', 'e0154',
        'e0155', 'e0159', 'e0161', 'e0163', 'e0166', 'e0170'
    ]
    return _load_physionet_db(record_ids, db_name="edb/1.0.0", source_fs=250)


def _load_physionet_db(record_ids, db_name, source_fs, lead_index=0):
    """
    Generic PhysioNet loader. Streams records directly from PhysioNet
    (no manual download needed in Colab — just needs internet).

    Args:
        record_ids : List of record name strings
        db_name    : PhysioNet database slug (used in the download path)
        source_fs  : Native sampling rate of this database
        lead_index : Which lead channel to use (0 = Lead I, 1 = Lead II)
    """
    all_segments, all_labels = [], []
    TARGET_FS = 360   # MIT-BIH standard

    for rec_id in record_ids:
        try:
            # Stream directly from PhysioNet
            record = wfdb.rdrecord(rec_id, pn_dir=db_name)
            ann    = wfdb.rdann(rec_id, 'atr', pn_dir=db_name)

            signal   = record.p_signal[:, lead_index]
            r_peaks  = ann.sample
            symbols  = ann.symbol

            segs, labs = extract_segments(
                signal, r_peaks, symbols, TARGET_FS, source_fs
            )

            if len(segs) > 0:
                all_segments.append(segs)
                all_labels.append(labs)

        except Exception as e:
            # Skip corrupt or missing records silently
            print(f"  Skipped {rec_id}: {e}")
            continue

    if not all_segments:
        print(f"  ⚠️  No segments loaded from {db_name}. "
              f"Check the pn_dir slug or your internet connection.")
        return np.empty((0, WINDOW), dtype=np.float32), np.empty((0,), dtype=np.int64)

    return (np.vstack(all_segments).astype(np.float32),
            np.concatenate(all_labels).astype(np.int64))


# ── Fusion augmentation ──────────────────────────────────────

def generate_fusion_samples(x_data, y_data, n_samples=500):
    """
    Creates synthetic Fusion beats by linearly interpolating between
    confirmed Normal and Ventricular beats from the same dataset.
    Physiologically grounded: Fusion IS a superposition of both.

    Args:
        x_data   : (N, 187) combined signal array
        y_data   : (N,) label array
        n_samples: How many Fusion samples to generate

    Returns:
        fusion_x (n_samples, 187), fusion_y (n_samples,) — all label 3
    """
    normal_beats = x_data[y_data == 0]
    ventr_beats  = x_data[y_data == 2]

    if len(normal_beats) == 0 or len(ventr_beats) == 0:
        print("⚠️  Need both Normal and Ventricular beats to generate Fusion.")
        return np.empty((0, WINDOW), dtype=np.float32), np.empty((0,), dtype=np.int64)

    fusion_x = []
    for _ in range(n_samples):
        n_beat = normal_beats[np.random.randint(len(normal_beats))]
        v_beat = ventr_beats[np.random.randint(len(ventr_beats))]
        alpha  = np.random.uniform(0.3, 0.7)   # blend ratio
        fusion_x.append(alpha * n_beat + (1 - alpha) * v_beat)

    fusion_x = np.array(fusion_x, dtype=np.float32)
    fusion_y = np.full(n_samples, 3, dtype=np.int64)
    return fusion_x, fusion_y


# ── Signal augmentation ──────────────────────────────────────

def augment_signals(x, y, multiplier=3):
    """
    Applies physiologically safe augmentations to rare class beats.
    Only augments classes 1, 2, 3 (Supraventricular, Ventricular, Fusion).
    Normal (class 0) is already majority — not augmented.

    Augmentations applied:
      - Amplitude scaling ±20%  (simulates electrode contact variation)
      - Baseline wander          (simulates breathing artifact)
      - Gaussian noise           (simulates electrode noise, σ=0.015)

    Args:
        x          : (N, 187) signal array
        y          : (N,) label array
        multiplier : How many augmented copies per original sample

    Returns:
        aug_x (M, 187), aug_y (M,) — augmented samples only (not originals)
        Append these to your existing arrays.
    """
    rare_mask = np.isin(y, [1, 2, 3])
    x_rare = x[rare_mask]
    y_rare = y[rare_mask]

    aug_x, aug_y = [], []
    t = np.linspace(0, 1, WINDOW)

    for _ in range(multiplier):
        for sig, label in zip(x_rare, y_rare):
            augmented = sig.copy()

            # 1. Amplitude scaling
            scale = np.random.uniform(0.80, 1.20)
            augmented = augmented * scale

            # 2. Baseline wander — low freq sine
            freq = np.random.uniform(0.05, 0.5)
            phase = np.random.uniform(0, 2 * np.pi)
            wander_amp = np.random.uniform(0.01, 0.05)
            augmented += wander_amp * np.sin(2 * np.pi * freq * t + phase)

            # 3. Gaussian noise
            augmented += np.random.normal(0, 0.015, WINDOW)

            # Re-clip to [0,1] after augmentation
            augmented = np.clip(augmented, 0.0, 1.0).astype(np.float32)

            aug_x.append(augmented)
            aug_y.append(label)

    return np.array(aug_x, dtype=np.float32), np.array(aug_y, dtype=np.int64)


# ── Master pipeline ──────────────────────────────────────────

def build_combined_dataset(x_train, y_train, augment_multiplier=3,
                            n_fusion_synthetic=500):
    """
    Full pipeline: pull external databases → augment → append to x_train.

    Args:
        x_train             : Your existing (N, 187) training array
        y_train             : Your existing (N,) label array
        augment_multiplier  : Augmented copies per rare-class beat
        n_fusion_synthetic  : Extra interpolated Fusion beats to generate

    Returns:
        x_combined (M, 187) float32, y_combined (M,) int64
        Ready to re-fit your scaler and retrain.
    """
    print("=" * 55)
    print("  DATASET EXPANSION PIPELINE")
    print("=" * 55)
    print(f"\nStarting size : {x_train.shape[0]} samples")
    print(f"Class dist    : {dict(sorted(Counter(y_train).items()))}\n")

    all_x = [x_train]
    all_y = [y_train]

    # ── Step 1: Pull external databases ───────────────────────
    print("── Step 1: Loading external PhysioNet databases")

    print("  Loading MIT-BIH SVA database...")
    x_sva, y_sva = load_mitbih_sva()
    print(f"  → {len(x_sva)} segments | {dict(sorted(Counter(y_sva).items()))}")

    print("  Loading INCART database...")
    x_inc, y_inc = load_incart()
    print(f"  → {len(x_inc)} segments | {dict(sorted(Counter(y_inc).items()))}")

    print("  Loading European ST-T database...")
    x_est, y_est = load_estdb()
    print(f"  → {len(x_est)} segments | {dict(sorted(Counter(y_est).items()))}")

    for xd, yd in [(x_sva, y_sva), (x_inc, y_inc), (x_est, y_est)]:
        if len(xd) > 0:
            all_x.append(xd)
            all_y.append(yd)

    # Pool everything so augmentation and fusion use the full set
    x_pool = np.vstack(all_x).astype(np.float32)
    y_pool = np.concatenate(all_y).astype(np.int64)

    # ── Step 2: Augmentation on rare classes ──────────────────
    print(f"\n── Step 2: Augmenting rare classes (×{augment_multiplier})")
    x_aug, y_aug = augment_signals(x_pool, y_pool, multiplier=augment_multiplier)
    print(f"  → {len(x_aug)} augmented samples added")

    # ── Step 3: Interpolated Fusion beats ─────────────────────
    print(f"\n── Step 3: Generating {n_fusion_synthetic} interpolated Fusion beats")
    x_fus, y_fus = generate_fusion_samples(x_pool, y_pool,
                                            n_samples=n_fusion_synthetic)
    print(f"  → {len(x_fus)} Fusion samples added")

    # ── Combine everything ─────────────────────────────────────
    x_combined = np.vstack([x_pool, x_aug, x_fus]).astype(np.float32)
    y_combined = np.concatenate([y_pool, y_aug, y_fus]).astype(np.int64)

    # Shuffle
    idx = np.random.permutation(len(x_combined))
    x_combined = x_combined[idx]
    y_combined = y_combined[idx]

    print(f"\n── Final dataset")
    print(f"  Total samples : {len(x_combined)}")
    print(f"  Class dist    : {dict(sorted(Counter(y_combined).items()))}")
    print("=" * 55)

    return x_combined, y_combined


In [ ]:
from sklearn.preprocessing import StandardScaler

x_combined, y_combined = build_combined_dataset(
    x_train,
    y_train,
    augment_multiplier=3,
    n_fusion_synthetic=500
)

scaler = StandardScaler()
x_combined_scaled = scaler.fit_transform(x_combined)

In [ ]:
from google.colab import drive
import numpy as np
import joblib

drive.mount('/content/drive')

save_path = '/content/drive/MyDrive/ecg_project/'
import os; os.makedirs(save_path, exist_ok=True)

# Save as compressed numpy — handles 881K samples efficiently
np.savez_compressed(
    save_path + 'combined_dataset.npz',
    x=x_combined,
    y=y_combined
)

# Also save the new scaler — critical, don't forget this
import joblib
joblib.dump(scaler, save_path + 'scaler_combined.pkl')

print("Saved.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import numpy as np
import joblib

data = np.load('/content/drive/MyDrive/ecg_project/combined_dataset.npz')
x_combined = data['x']
y_combined = data['y']
scaler = joblib.load('/content/drive/MyDrive/ecg_project/scaler_combined.pkl')

In [ ]:
df = pd.DataFrame(x_combined)
df['label'] = y_combined

In [ ]:
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values.astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

df_train = pd.DataFrame(X_train)
df_train['label'] = y_train

df_test = pd.DataFrame(X_test)
df_test['label'] = y_test

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
print("\n Class Distribution in Training Set ")
class_names = ['Normal', 'Supraventricular', 'Ventricular', 'Fusion', 'Unclassifiable']
for i in range(5):
  count = np.sum(y_train == i)
  percentage = 100 * count / len(y_train)
  print(f"Class {i} ({class_names[i]}): {count} ({percentage:.2f}%)")

print("\n Class Distribution in Test Set")
for i in range(5):
  count = np.sum(y_test == i)
  percentage = 100 * count / len(y_test)
  print(f"Class {i} ({class_names[i]}): {count} ({percentage:.2f}%)")

In [ ]:
print("\n Data Quality analysis ")
print(f"Training set - Missing values: {df_train.isnull().sum().sum()}")
print(f"Test set - Missing values: {df_test.isnull().sum().sum()}")

if df_train.isnull().sum().sum() > 0:
  print("Warning: Found missung values in training data")
  print(df_train.isnull().sum()[df_train.isnull().sum() > 0])

if df_test.isnull().sum().sum() > 0:
  print("Warning: Found missung values in test data")
  print(df_test.isnull().sum()[df_test.isnull().sum() > 0])

In [ ]:
print("\n Duplicate check")
train_duplicate = df_train.duplicated().sum()
test_duplicate = df_test.duplicated().sum()
print(f"Training set - Duplicate rows: {train_duplicate} ({100*train_duplicate/len(df_train):.2f}%)")
print(f"Test set - Duplicate rows: {test_duplicate} ({100*test_duplicate/len(df_train):.2f}%)")

In [ ]:
if train_duplicate > 0:
  print(f"\n Removing {train_duplicate} rows from training data.")
  df_train = df_train.drop_duplicates()
  X_train = df_train.iloc[:, :-1].values
  y_train = df_train.iloc[:, -1].values.astype(int)
  print(f"New training set shape: {df_train.shape}")

if test_duplicate > 0:
  print(f"\n Removing {test_duplicate} rows from test data.")
  df_test = df_test.drop_duplicates()
  X_test = df_test.iloc[:, :-1].values
  y_test = df_test.iloc[:, -1].values.astype(int)
  print(f"New testing set shape: {df_test.shape}")

In [ ]:
print("\n Infinite values check:")
train_inf = np.isinf(X_train).sum()
print(f"Infinite values in training data: {train_inf}")

test_inf = np.isinf(X_test).sum()
print(f"Infinite values in test data: {test_inf}")

if train_inf > 0 or test_inf > 0:
  print("Warning: found infinite values")

In [ ]:
print("\n Data Range Analysis: ")
print(f"Training set = Min: {X_train.min():.4f}, Max: {X_train.max():.4f}, Mean: {X_train.mean():.4f}, Std: {X_train.std():.4f}")
print(f"Test set = Min: {X_test.min():.4f}, Max: {X_test.max():.4f}, Mean: {X_test.mean():.4f}, Std: {X_test.std():.4f}")

In [ ]:
print("\n Label Validation")
unique_train_labels = np.unique(y_train)
unique_test_labels = np.unique(y_test)
print(f"\nTraining set - Unique labels: {unique_train_labels}")
print(f"Test set - Unique labels: {unique_test_labels}")

if not np.array_equal(unique_train_labels, np.array([0,1,2,3,4])):
  print("Warning: Unexpected labels in training data")

if not np.array_equal(unique_test_labels, np.array([0,1,2,3,4])):
  print("Warning: Unexpected labels in test data")

In [ ]:
print("\n=== Class Distribution (Imbalance Analysis) ===")
class_names = ['Normal', 'Supraventricular', 'Ventricular', 'Fusion', 'Unclassifiable']

train_counts = [np.sum(y_train == i) for i in range(5)]
test_counts = [np.sum(y_test == i) for i in range(5)]

print("\nTraining Set:")
for i in range(5):
    count = train_counts[i]
    percentage = 100 * count / len(y_train)
    print(f"  Class {i} ({class_names[i]:20s}): {count:6d} samples ({percentage:5.2f}%)")

print("\nTest Set:")
for i in range(5):
    count = test_counts[i]
    percentage = 100 * count / len(y_test)
    print(f"  Class {i} ({class_names[i]:20s}): {count:6d} samples ({percentage:5.2f}%)")

In [ ]:
max_class_count = max(train_counts)
min_class_count = min(train_counts)
imbalance_ratio = max_class_count / min_class_count
print(f"\nImbalance Ratio: {imbalance_ratio:.2f}:1")
if imbalance_ratio > 10:
    print("Dataset is HIGHLY IMBALANCED - Class weighting will be applied")

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(range(5), train_counts, color='steelblue', edgecolor='black', alpha=0.8)
plt.xlabel('Class', fontsize=12)
plt.ylabel('Number of Samples', fontsize=12)
plt.title('Class Distribution - Training Set', fontsize=14, fontweight='bold')
plt.xticks(range(5), [f'Class {i}\n{class_names[i]}' for i in range(5)])
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import plotly.express as px

labels = {
    0: "Normal",
    1: "Superventricular Contraction",
    2: "Premature Ventricular Contraction",
    3: "Fusion of ventricular and normal contraction",
    4: "Unclassifiable beat"
}

value_counts = df_train.iloc[:, -1].value_counts().sort_index().rename(labels)

bar_fig = px.bar(x=value_counts.index, y=value_counts.values,
                 labels={'x': 'Heartbeat Type', 'y': 'Number of Samples'},
                 text_auto=True,
                 title="Distribution of Heartbeat Types in Training Dataset",
                 color=value_counts.values,
                 color_continuous_scale='Blues')

pie_fig = px.pie(names=value_counts.index, values=value_counts.values,
                 title="Percentage Distribution of Heartbeat Types in Training Dataset",
                 hole=0.3)
bar_fig.update_layout(title_x=0.5, width=900, height=600, showlegend=False)
pie_fig.update_layout(title_x=0.5, width=900, height=600)

bar_fig.show()
pie_fig.show()


In [ ]:
class ECGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X).unsqueeze(1)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
train_dataset = ECGDataset(X_train_scaled, y_train)
test_dataset = ECGDataset(X_test_scaled, y_test)

In [ ]:
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = batch_size, shuffle = False)

In [4]:
class CNN_LSTM(nn.Module):
    """
    Hybrid CNN-LSTM architecture for ECG signal classification

    CNN layers: Extract spatial features from signals
    LSTM layers: Capture temporal dependencies
    """
    def __init__(self, n_classes=5, input_channels=1, input_length=187):
        super(CNN_LSTM, self).__init__()

        # CNN blocks
        # Conv Block 1
        self.conv1 = nn.Conv1d(input_channels, 32, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(32)
        self.pool1 = nn.MaxPool1d(2)
        self.dropout1 = nn.Dropout(0.2)

        # Conv Block 2
        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool2 = nn.MaxPool1d(2)
        self.dropout2 = nn.Dropout(0.2)

        # Conv Block 3
        self.conv3 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(2)
        self.dropout3 = nn.Dropout(0.2)

        # Calculate size after convolutions
        # After 3 MaxPool layers with kernel_size=2: input_length // (2^3)
        lstm_input_size = input_length // 8

        # LSTM layers
        self.lstm = nn.LSTM(
            input_size=128,  # number of CNN feature maps
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
            bidirectional=True  # Bidirectional LSTM for better performance
        )

        # Fully Connected layers
        # Bidirectional LSTM outputs 2 * hidden_size
        self.fc1 = nn.Linear(128, 64)  # 64 * 2 = 128
        self.dropout4 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(64, n_classes)

    def forward(self, x):
        # CNN part
        # x shape: (batch, 1, length)
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool1(x)
        x = self.dropout1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool2(x)
        x = self.dropout2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool3(x)
        x = self.dropout3(x)

        # Prepare for LSTM: (batch, channels, length) -> (batch, length, channels)
        x = x.permute(0, 2, 1)

        # LSTM part
        # x shape: (batch, seq_len, features)
        lstm_out, (hidden, cell) = self.lstm(x)

        # Take last output from LSTM
        x = lstm_out[:, -1, :]

        # Fully connected
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout4(x)
        x = self.fc2(x)

        return x

# Instantiate model
model = CNN_LSTM(n_classes=5, input_channels=1, input_length=187)
model = model.to(device)

# Display architecture
print(model)
print(f'\nTotal number of parameters: {sum(p.numel() for p in model.parameters())}')

CNN_LSTM(
  (conv1): Conv1d(1, 32, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout1): Dropout(p=0.2, inplace=False)
  (conv2): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout2): Dropout(p=0.2, inplace=False)
  (conv3): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn3): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout3): Dropout(p=0.2, inplace=False)
  (lstm): LSTM(128, 64, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (fc1): Linear(in_features=12

In [5]:
class_counts = np.bincount(y_train, minlength=5)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * len(class_counts)
class_weights_tensor = torch.FloatTensor(class_weights).to(device)

print(f"\nClass weights for loss function:")
for i, weight in enumerate(class_weights):
    print(f" Class {i} ({class_names[i]}): {weight:.4f}")

NameError: name 'y_train' is not defined

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

In [ ]:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """Train model for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

def validate(model, dataloader, criterion, device):
    """Validate model"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = running_loss / len(dataloader)
    val_acc = 100 * correct / total
    return val_loss, val_acc

In [ ]:
num_epochs = 50
train_losses, train_accs = [], []
val_losses, val_accs = [], []
best_val_acc = 0.0

print("\nStarting training...\n")
print(f"{'Epoch':<8} {'Train Loss':<12} {'Train Acc':<12} {'Val Loss':<12} {'Val Acc':<12}")
print("="*60)

for epoch in range(num_epochs):
    # Training
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)

    # Validation
    val_loss, val_acc = validate(model, test_loader, criterion, device)

    # Save history
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    # Learning rate scheduler
    scheduler.step(val_loss)

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_ecg_model.pth')
        print(f'{epoch+1:<8} {train_loss:<12.4f} {train_acc:<12.2f}% {val_loss:<12.4f} {val_acc:<12.2f}% ⭐ BEST')
    else:
        print(f'{epoch+1:<8} {train_loss:<12.4f} {train_acc:<12.2f}% {val_loss:<12.4f} {val_acc:<12.2f}%')

print("\n" + "="*60)
print(f'Training complete! Best validation accuracy: {best_val_acc:.2f}%')
print("="*60)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/ecg_models/
!cp /content/best_ecg_model.pth /content/drive/MyDrive/ecg_models/best_model.pt

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SAVE MODEL PROPERLY (For use in Google Colab)
# ═══════════════════════════════════════════════════════════════

import torch
import shutil
import os

# Create checkpoint with everything needed
checkpoint = {
    'model_state_dict': model.state_dict(),
    'model_architecture': {
        'n_classes': 5,
        'input_channels': 1,
        'input_length': 187
    },
    'scaler': scaler,
    'class_names': ['Normal', 'Supraventricular', 'Ventricular', 'Fusion', 'Unclassifiable'],
    'training_info': {
        'best_accuracy': best_val_acc,
        'num_parameters': sum(p.numel() for p in model.parameters()),
        'device': str(device)
    }
}

# Save locally in Colab
torch.save(checkpoint, 'ecg_model_checkpoint.pth')
print("✓ Checkpoint saved locally")

# Upload to Google Drive
from google.colab import drive
drive.mount('/content/drive')

os.makedirs('/content/drive/MyDrive/ecg_models', exist_ok=True)
shutil.copy('ecg_model_checkpoint.pth', 
            '/content/drive/MyDrive/ecg_models/ecg_model_checkpoint.pth')

# Also save state dict separately (backup approach)
torch.save(model.state_dict(), 'best_ecg_model.pth')
shutil.copy('best_ecg_model.pth',
            '/content/drive/MyDrive/ecg_models/best_ecg_model.pth')

# Save scaler
import joblib
joblib.dump(scaler, 'scaler_combined.pkl')
shutil.copy('scaler_combined.pkl',
            '/content/drive/MyDrive/ecg_models/scaler_combined.pkl')

print("✓ All files uploaded to Google Drive")
print("\nFiles saved to: /MyDrive/ecg_models/")
print("  - ecg_model_checkpoint.pth (RECOMMENDED)")
print("  - best_ecg_model.pth (Backup)")
print("  - scaler_combined.pkl (Backup)")

In [ ]:
fig, (ax1, ax2) = plt

In [ ]:
print("\n" + "="*70)
print("FINAL MODEL EVALUATION")
print("="*70)

# Load best model
model.load_state_dict(torch.load('best_ecg_model.pth'))
model.eval()

y_true = []
y_pred = []
y_probs = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        probabilities = F.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs.data, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())
        y_probs.extend(probabilities.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_probs = np.array(y_probs)

overall_acc = accuracy_score(y_true, y_pred) * 100
print(f"OVERALL TEST ACCURACY: {overall_acc:.2f}%\n")

print("="*70)
print("PER-CLASS PERFORMANCE")
print("="*70)
for i in range(5):
    class_mask = y_true == i
    class_acc = accuracy_score(y_true[class_mask], y_pred[class_mask]) * 100
    class_total = np.sum(class_mask)
    class_correct = np.sum((y_true[class_mask] == y_pred[class_mask]))
    print(f"{class_names[i]:30s}: {class_correct:4d}/{class_total:4d} correct ({class_acc:6.2f}%)")

print("\n" + "="*70)
print("CONFUSION MATRIX")
print("="*70)
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - Test Set', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
print("\n" + "="*70)
print("DETAILED CLASSIFICATION REPORT")
print("="*70)
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

from sklearn.metrics import precision_recall_fscore_support, cohen_kappa_score

precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, average='weighted')
kappa = cohen_kappa_score(y_true, y_pred)

print("\n" + "="*70)
print("AGGREGATE METRICS")
print("="*70)
print(f"Weighted Precision: {precision*100:.2f}%")
print(f"Weighted Recall:    {recall*100:.2f}%")
print(f"Weighted F1-Score:  {f1*100:.2f}%")
print(f"Cohen's Kappa:      {kappa:.4f}")

In [11]:
def predict_ecg(model, signal, scaler, device):
    """
    Prediction on a single ECG signal

    Args:
        model: Trained PyTorch model
        signal: NumPy array of ECG signal (1D)
        scaler: StandardScaler used for normalization
        device: CPU or GPU

    Returns:
        predicted_class, probabilities
    """
    model.eval()

    # Normalize signal
    signal_scaled = scaler.transform(signal.reshape(1, -1))

    # Convert to tensor
    signal_tensor = torch.FloatTensor(signal_scaled).unsqueeze(1).to(device)

    with torch.no_grad():
        output = model(signal_tensor)
        probabilities = F.softmax(output, dim=1)
        predicted_class = torch.argmax(probabilities, dim=1).item()

    return predicted_class, probabilities.cpu().numpy()[0]

In [9]:
# Example prediction
sample_signal = X_test[0]
pred_class, probs = predict_ecg(model, sample_signal, scaler, device)
print(f"\nExample prediction:")
print(f"Predicted class: {pred_class} ({class_names[pred_class]})")
print(f"Probabilities:")
for i, prob in enumerate(probs):
    print(f"  {class_names[i]}: {prob*100:.2f}%")
print(f"True class: {y_test[0]} ({class_names[y_test[0]]})")

NameError: name 'X_test' is not defined

In [ ]:
print("\n" + "="*70)
print("EXAMPLE PREDICTION WITH VISUALIZATION")
print("="*70)

sample_idx = 0
sample_signal = X_test[sample_idx]
true_label = y_test[sample_idx]

# Make prediction
pred_class, probs = predict_ecg(model, sample_signal, scaler, device)

# Create visualization
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8))

# Top plot - ECG Signal
ax1.plot(sample_signal, linewidth=2, color='darkblue')
ax1.set_title(f'ECG Signal Sample #{sample_idx}', fontsize=14, fontweight='bold')
ax1.set_xlabel('Time Steps', fontsize=12)
ax1.set_ylabel('Amplitude', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 186)

# Add prediction and true label as text on the plot
textstr = f'True Class: {true_label} ({class_names[true_label]})\nPredicted: {pred_class} ({class_names[pred_class]})'
props = dict(boxstyle='round', facecolor='wheat' if pred_class == true_label else 'lightcoral', alpha=0.8)
ax1.text(0.02, 0.98, textstr, transform=ax1.transAxes, fontsize=11,
         verticalalignment='top', bbox=props)

# Bottom plot - Prediction Probabilities
colors_bar = ['green' if i == pred_class else 'steelblue' for i in range(5)]
bars = ax2.bar(range(5), probs * 100, color=colors_bar, edgecolor='black', alpha=0.7)
ax2.set_xlabel('Class', fontsize=12)
ax2.set_ylabel('Probability (%)', fontsize=12)
ax2.set_title('Model Confidence for Each Class', fontsize=14, fontweight='bold')
ax2.set_xticks(range(5))
ax2.set_xticklabels([f'{i}\n{class_names[i]}' for i in range(5)], rotation=0)
ax2.set_ylim(0, 105)
ax2.grid(axis='y', alpha=0.3)

# Add percentage labels on bars
for i, (bar, prob) in enumerate(zip(bars, probs)):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
             f'{prob*100:.2f}%',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

# Print detailed results
print(f"\n{'='*70}")
print(f"Sample Index: {sample_idx}")
print(f"True Class: {true_label} ({class_names[true_label]})")
print(f"Predicted Class: {pred_class} ({class_names[pred_class]})")
print(f"Prediction Status: {'CORRECT' if pred_class == true_label else '❌ INCORRECT'}")
print(f"\nProbability Distribution:")
for i, prob in enumerate(probs):
    marker = "👉" if i == pred_class else "  "
    print(f"{marker} {class_names[i]:30s}: {prob*100:6.2f}%")
print(f"{'='*70}")

In [ ]:
import matplotlib.gridspec as gridspec
from collections import defaultdict

# MIT-BIH class labels (standard 5-class mapping)
CLASS_NAMES = {
    0: "Normal",
    1: "Supraventricular",
    2: "Ventricular",
    3: "Fusion",
    4: "Unclassifiable"
}

RARE_CLASSES = [1, 2, 3]  # The unusual categories we care about


# ============================================================
# PART 1 — RARE PATTERN STRESS TEST
# ============================================================

def stress_test_rare_classes(model, x_test, y_test, scaler, device, n_samples=20):
    """
    Pull real rare-class samples from your test set and evaluate model
    confidence. Shows per-class accuracy and confidence distribution.

    Args:
        model     : Trained PyTorch model
        x_test    : NumPy array of test signals (N x 187)
        y_test    : NumPy array of test labels (N,)
        scaler    : StandardScaler used during training
        device    : torch device
        n_samples : How many samples to test per rare class
    """
    model.eval()
    results = defaultdict(list)

    for target_class in RARE_CLASSES:
        # Find indices in test set belonging to this class
        indices = np.where(y_test == target_class)[0]

        if len(indices) == 0:
            print(f"No samples found for class {target_class} "
                  f"({CLASS_NAMES[target_class]}) in test set.")
            continue

        # Sample up to n_samples (or all available if fewer)
        chosen = np.random.choice(indices, size=min(n_samples, len(indices)),
                                  replace=False)

        for idx in chosen:
            signal = x_test[idx]
            true_label = y_test[idx]
            pred_class, probs = predict_ecg(model, signal, scaler, device)

            results[target_class].append({
                "true": true_label,
                "pred": pred_class,
                "correct": pred_class == true_label,
                "confidence": probs[pred_class],        # confidence in predicted class
                "true_class_prob": probs[true_label],   # prob assigned to true class
                "all_probs": probs
            })

    # ── Print summary ──────────────────────────────────────────
    print("\n" + "="*60)
    print("  RARE CLASS STRESS TEST RESULTS")
    print("="*60)

    for cls in RARE_CLASSES:
        if cls not in results:
            continue
        entries = results[cls]
        n = len(entries)
        n_correct = sum(e["correct"] for e in entries)
        avg_conf = np.mean([e["confidence"] for e in entries])
        avg_true_prob = np.mean([e["true_class_prob"] for e in entries])

        print(f"\n▸ Class {cls}: {CLASS_NAMES[cls]}  (n={n})")
        print(f"  Accuracy              : {n_correct}/{n} = {n_correct/n*100:.1f}%")
        print(f"  Avg confidence (pred) : {avg_conf*100:.1f}%")
        print(f"  Avg prob on true class: {avg_true_prob*100:.1f}%")

        # Show misclassified cases
        misses = [e for e in entries if not e["correct"]]
        if misses:
            print(f"  Misclassified as:")
            confusion = defaultdict(int)
            for m in misses:
                confusion[CLASS_NAMES[m["pred"]]] += 1
            for wrong_class, count in sorted(confusion.items(),
                                              key=lambda x: -x[1]):
                print(f"    → {wrong_class}: {count}x")

    # ── Visualise probability distributions ───────────────────
    _plot_stress_test_results(results)
    return results


def _plot_stress_test_results(results):
    """Bar chart of average class probabilities per rare class."""
    n_rare = len([c for c in RARE_CLASSES if c in results])
    if n_rare == 0:
        return

    fig, axes = plt.subplots(1, n_rare, figsize=(5 * n_rare, 4))
    if n_rare == 1:
        axes = [axes]

    colors = ["#4CAF50" if i == j else "#EF5350"
              for i in range(5) for j in range(5)]

    for ax, cls in zip(axes, [c for c in RARE_CLASSES if c in results]):
        entries = results[cls]
        avg_probs = np.mean([e["all_probs"] for e in entries], axis=0)

        bar_colors = ["#4CAF50" if i == cls else "#90CAF9"
                      for i in range(len(CLASS_NAMES))]
        bars = ax.bar(list(CLASS_NAMES.values()), avg_probs * 100,
                      color=bar_colors, edgecolor="white", linewidth=0.8)

        ax.set_title(f"True: {CLASS_NAMES[cls]}", fontsize=12, fontweight="bold")
        ax.set_ylabel("Avg Probability (%)")
        ax.set_ylim(0, 100)
        ax.tick_params(axis="x", rotation=30)

        # Annotate bars
        for bar, p in zip(bars, avg_probs):
            if p > 0.01:
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + 1,
                        f"{p*100:.1f}%", ha="center", va="bottom", fontsize=8)

    plt.suptitle("Model Confidence on Rare ECG Classes", fontsize=14,
                 fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig("stress_test_results.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("\n Plot saved → stress_test_results.png")

In [ ]:
# Run the stress test on rare classes
stress_test_rare_classes(model, X_test, y_test, scaler, device, n_samples=20)

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import cv2
from scipy.signal import resample
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict

def ecg_image_to_signal(image_path, target_length=187, debug=False):
    # ── 1. Load & greyscale ────────────────────────────────────
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Could not load image: {image_path}")

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape

    # ── 2. Threshold — isolate the dark waveform ───────────────
    # Otsu's method works well on clean black-on-white ECG images
    _, binary = cv2.threshold(gray, 0, 255,
                               cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Optional: remove thin noise (1-px specks) while keeping waveform
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 2))
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

    # ── 3. Column-wise Y centroid extraction ───────────────────
    raw_signal = np.full(w, np.nan)

    for col in range(w):
        column_pixels = binary[:, col]
        dark_rows = np.where(column_pixels > 0)[0]

        if len(dark_rows) == 0:
            continue  # gap — will interpolate later

        # Centroid (handles thick lines better than simple min/max)
        raw_signal[col] = np.mean(dark_rows)

    # ── 4. Fill gaps via linear interpolation ─────────────────
    nan_mask = np.isnan(raw_signal)
    if nan_mask.all():
        raise ValueError(
            "No waveform detected. Check that the image has a dark trace "
            "on a light background, and isn't inverted."
        )

    x_valid = np.where(~nan_mask)[0]
    y_valid = raw_signal[~nan_mask]
    raw_signal = np.interp(np.arange(w), x_valid, y_valid)

    # ── 5. Invert (high Y = bottom of image = low voltage) ────
    #       then normalise to [0, 1]
    inverted = (h - 1) - raw_signal
    signal_norm = (inverted - inverted.min()) / (inverted.max() - inverted.min() + 1e-8)

    # ── 6. Resample to 187 ────────────────────────────────────
    signal_187 = resample(signal_norm, target_length)

    # Clip to [0,1] in case resampling introduced tiny overshoots
    signal_187 = np.clip(signal_187, 0.0, 1.0)

    # ── Debug plot ─────────────────────────────────────────────
    if debug:
        _plot_extraction(img, binary, raw_signal, signal_187, h, w)

    return signal_187.astype(np.float32)


def _plot_extraction(img, binary, raw_signal, signal_187, h, w):
    fig = plt.figure(figsize=(14, 8))
    gs = gridspec.GridSpec(2, 3, figure=fig)

    # Original image
    ax1 = fig.add_subplot(gs[0, :2])
    ax1.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax1.set_title("1. Original ECG Image", fontsize=11)
    ax1.axis("off")

    # Binary (thresholded)
    ax2 = fig.add_subplot(gs[0, 2])
    ax2.imshow(binary, cmap="gray")
    ax2.set_title("2. Thresholded (Otsu)", fontsize=11)
    ax2.axis("off")

    # Waveform overlay on original
    ax3 = fig.add_subplot(gs[1, :2])
    ax3.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), aspect="auto")
    ax3.plot(np.arange(w), raw_signal, color="red", linewidth=1.5,
             label="Extracted trace", alpha=0.85)
    ax3.set_title("3. Extracted Waveform Overlay", fontsize=11)
    ax3.legend(fontsize=9)
    ax3.axis("off")

    # Final 187-point signal
    ax4 = fig.add_subplot(gs[1, 2])
    ax4.plot(signal_187, color="#1565C0", linewidth=1.8)
    ax4.fill_between(range(len(signal_187)), signal_187, alpha=0.15,
                     color="#1565C0")
    ax4.set_title(f"4. Resampled Signal (187 pts)", fontsize=11)
    ax4.set_xlabel("Sample index")
    ax4.set_ylabel("Normalised amplitude")
    ax4.grid(True, alpha=0.3)

    plt.suptitle("ECG Image → Signal Extraction Pipeline",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig("extraction_debug.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Debug plot saved → extraction_debug.png")


def predict_from_image(image_path, model, scaler, device,
                       target_length=187, debug=False):
    print(f"\nLoading image: {image_path}")

    # Step 1: Extract signal from image
    signal = ecg_image_to_signal(image_path, target_length=target_length,
                                  debug=debug)
    print(f"Signal extracted — shape: {signal.shape}, "
          f"range: [{signal.min():.3f}, {signal.max():.3f}]")

    # Step 2: Predict
    pred_class, probs = predict_ecg(model, signal, scaler, device)

    # Step 3: Pretty-print results
    print("\n" + "─"*45)
    print("  ECG CLASSIFICATION RESULT")
    print("─"*45)
    print(f"  Predicted class : {pred_class} → {CLASS_NAMES[pred_class]}")
    print(f"  Confidence      : {probs[pred_class]*100:.2f}%")
    print("\n  Full probability breakdown:")
    for i, p in enumerate(probs):
        bar = "█" * int(p * 30)
        marker = " -> predicted" if i == pred_class else ""
        print(f"    {CLASS_NAMES[i]:<18} {p*100:5.2f}%  {bar}{marker}")
    print("─"*45)

    return pred_class, probs, signal

In [ ]:
pred_class, probs, signal = predict_from_image(
    image_path="/content/normal.png",
    model=model, scaler=scaler, device=device,
    debug=True
)

In [ ]:
pred_class, probs, signal = predict_from_image(
    image_path="/content/supraventricular.png",
    model=model, scaler=scaler, device=device,
    debug=True
)

In [ ]:
pred_class, probs, signal = predict_from_image(
    image_path="/content/ventricular.png",
    model=model, scaler=scaler, device=device,
    debug=True
)

In [ ]:
pred_class, probs, signal = predict_from_image(
    image_path="/content/fusion.png",
    model=model, scaler=scaler, device=device,
    debug=True
)

In [8]:
# ═══════════════════════════════════════════════════════════════
# LOAD MODEL IN VSCode (For use on local system)
# ═══════════════════════════════════════════════════════════════

import torch
import joblib
import os

print(f"Current working directory: {os.getcwd()}")
print(f"Files present: {os.listdir('.')}\n")

# ── OPTION A: Load from Checkpoint (RECOMMENDED) ────────────────
checkpoint_path = 'ecg_model_checkpoint.pth'

if os.path.exists(checkpoint_path):
    print(f"✓ Found checkpoint file: {checkpoint_path}\n")
    
    # Determine device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"✓ Device: {device}\n")
    
    # Load checkpoint (weights_only=False for scikit-learn objects)
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    print("✓ Checkpoint loaded")
    
    # Recreate model from architecture specs in checkpoint
    model_arch = checkpoint['model_architecture']
    model = CNN_LSTM(
        n_classes=model_arch['n_classes'],
        input_channels=model_arch['input_channels'],
        input_length=model_arch['input_length']
    )
    print("✓ Model architecture created")
    
    # Load trained weights
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    print("✓ Model weights loaded and set to eval mode")
    
    # Extract scaler and class names
    scaler = checkpoint['scaler']
    class_names = checkpoint['class_names']
    
    # Extract optional training info
    training_info = checkpoint.get('training_info', {})
    best_val_acc = training_info.get('best_accuracy', 'N/A')
    num_params = training_info.get('num_parameters', 'N/A')
    
    print(f"\n{'='*60}")
    print(f"✅ MODEL LOADED SUCCESSFULLY!")
    print(f"{'='*60}")
    print(f"Architecture: CNN-LSTM")
    print(f"Classes: {len(class_names)} classes")
    for i, name in enumerate(class_names):
        print(f"  {i}: {name}")
    if isinstance(best_val_acc, (int, float)):
        print(f"Best Training Accuracy: {best_val_acc:.2f}%")
    if isinstance(num_params, int):
        print(f"Total Parameters: {num_params:,}")
    print(f"Device: {device}")
    print(f"{'='*60}\n")

else:
    # ── OPTION B: Load State Dict + Scaler (Backup Approach) ────
    state_dict_path = 'best_ecg_model.pth'
    scaler_path = 'scaler_combined.pkl'
    
    if os.path.exists(state_dict_path) and os.path.exists(scaler_path):
        print(f"Checkpoint not found. Using backup files...\n")
        
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        # Create model
        model = CNN_LSTM(n_classes=5, input_channels=1, input_length=187)
        
        # Load weights
        model.load_state_dict(torch.load(state_dict_path, map_location=device, weights_only=True))
        model.to(device)
        model.eval()
        
        # Load scaler
        scaler = joblib.load(scaler_path)
        class_names = ['Normal', 'Supraventricular', 'Ventricular', 'Fusion', 'Unclassifiable']
        
        print(f"✓ Model loaded from state dict")
        print(f"✓ Scaler loaded")
        print(f"Device: {device}\n")
    
    else:
        print("❌ ERROR: No model files found!")
        print(f"   Expected: {checkpoint_path} or {state_dict_path}")
        print("\nDownload from Google Drive:")
        print("  1. Go to https://drive.google.com/drive/folders/YOUR_FOLDER_ID")
        print("  2. Download ecg_model_checkpoint.pth")
        print("  3. Place in: c:\\Users\\shahe\\OneDrive\\Documents\\Desktop\\ECG\\")
        raise FileNotFoundError("Model files not found")

Current working directory: c:\Users\shahe\OneDrive\Documents\Desktop\ECG
Files present: ['best_model.pt', 'DEPLOYMENT_WORKFLOW.md', 'desktop.ini', 'ecg', 'ECG_Arrhythmia.ipynb', 'ecg_model_checkpoint.pth', 'ecg_model_loader.py', 'ecg_ui.py', 'kaggle.json', 'MODEL_SAVE_LOAD_GUIDE.md', 'QUICK_CHECKLIST.txt', 'QUICK_REFERENCE.md', 'README_DOCUMENTATION.md', 'requirements.txt', 'scaler_combined.pkl', 'START_HERE.txt']

✓ Found checkpoint file: ecg_model_checkpoint.pth

✓ Device: cpu

✓ Checkpoint loaded
✓ Model architecture created
✓ Model weights loaded and set to eval mode

✅ MODEL LOADED SUCCESSFULLY!
Architecture: CNN-LSTM
Classes: 5 classes
  0: Normal
  1: Supraventricular
  2: Ventricular
  3: Fusion
  4: Unclassifiable
Device: cpu



In [14]:
import streamlit as st

def load_model_into_session(model, scaler, device):
    """Call this once after training to make the model available to the UI."""
    st.session_state["model"]  = model
    st.session_state["scaler"] = scaler
    st.session_state["device"] = device
    print("Model loaded into session state.")

# After training:
load_model_into_session(model, scaler, device)

# Then launch the UI:
# !streamlit run ecg_ui.py &
        

2026-05-03 23:47:41.374 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 23:47:41.386 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`
2026-05-03 23:47:41.386 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 23:47:41.390 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 23:47:41.390 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 23:47:41.390 WARNING streamlit.runtime.scriptrunner_utils.script_run_c

Model loaded into session state.


In [13]:
# Load your signal (187 points)
signal = X_test[0]  # or any ECG signal

# Get prediction
pred_class, probs = predict_ecg(model, signal, scaler, device)

# Display results
print(f"Prediction: {class_names[pred_class]}")
print(f"Confidence: {probs[pred_class]*100:.2f}%")

NameError: name 'X_test' is not defined